# Performance Benchmark

This notebook measures how long a full CPM recompute takes on a large,
worst-case project, and checks that repeated runs are deterministic.
`schedule()` always recomputes the whole project; there is no incremental
mode.

**Install**
```bash
pip install trueppm-scheduler
```

In [ ]:
import time
from datetime import date, timedelta

from trueppm_scheduler import (
    Calendar,
    Dependency,
    Project,
    Task,
    schedule,
)

## 1. Build a medium-sized FS chain project

500 tasks in a single dependency chain is a worst-case scenario for the
forward pass (no parallelism). We use this to measure full-recompute timing.

In [ ]:
def make_fs_chain(n: int, start: date = date(2026, 1, 5)) -> Project:
    """Build an n-task FS chain — worst-case for forward-pass traversal."""
    tasks = [Task(id=f"t{i}", name=f"Task {i}", duration=timedelta(days=1)) for i in range(n)]
    deps = [Dependency(predecessor_id=f"t{i}", successor_id=f"t{i + 1}") for i in range(n - 1)]
    return Project(
        id="bench",
        name=f"{n}-task chain",
        start_date=start,
        tasks=tasks,
        dependencies=deps,
        calendar=Calendar(),
    )


project = make_fs_chain(500)
print(f"Project: {len(project.tasks)} tasks, {len(project.dependencies)} dependencies")

## 2. Full recompute timing

How long does a full recompute take on this 500-task chain?
The `scheduler:bench` CI job guards a target of < 2 000 ms.

In [ ]:
# Warm up (import-time costs)
schedule(project)

# Full recompute
t0 = time.perf_counter()
result_full = schedule(project)
full_ms = (time.perf_counter() - t0) * 1000

print(f"Full recompute: {full_ms:.1f} ms  (target: < 2000 ms)")

## 3. Result consistency

Running a full recompute twice must produce identical results.

In [ ]:
result_full2 = schedule(project)

mismatches = []
for tf, tf2 in zip(result_full.tasks, result_full2.tasks, strict=True):
    if tf.early_start != tf2.early_start or tf.early_finish != tf2.early_finish:
        mismatches.append(tf.id)

if mismatches:
    print(f"MISMATCH on tasks: {mismatches}")
else:
    print("Full recompute is deterministic: both runs agree on all task dates ✓")

assert result_full.project_finish == result_full2.project_finish
print(f"Project finish: {result_full.project_finish}")

## 4. Performance target and CI guard

| Scenario | Target |
|----------|--------|
| 500-task chain, full recompute | < 2 000 ms |

The `scheduler:bench` CI job in `.gitlab-ci.yml` enforces this limit.

```bash
# Run locally
cd packages/scheduler
pytest tests/test_bench.py -v
```

In [ ]:
# Quick local bench: repeated full recomputes
import statistics

RUNS = 10
full_times = []

for _ in range(RUNS):
    t0 = time.perf_counter()
    schedule(project)
    full_times.append((time.perf_counter() - t0) * 1000)

p95_idx = max(0, int(RUNS * 0.95) - 1)
print(
    f"Full recompute — median {statistics.median(full_times):.1f} ms, "
    f"p95 {sorted(full_times)[p95_idx]:.1f} ms  (target: < 2000 ms)"
)